In [1]:
!pip install chumpy git+https://github.com/vchoutas/smplx timm einops wandb yacs trimesh pyrender ultralytics fastapi uvicorn pyngrok nest-asyncio
!pip install pytorch_lightning

!git clone https://github.com/rolpotamias/WiLoR.git
!mkdir -p pretrained_models
!mkdir -p mano_data

!wget https://huggingface.co/spaces/rolpotamias/WiLoR/resolve/main/pretrained_models/detector.pt -O pretrained_models/detector.pt
!wget https://huggingface.co/spaces/rolpotamias/WiLoR/resolve/main/pretrained_models/wilor_final.ckpt -O pretrained_models/wilor_final.ckpt
!wget https://huggingface.co/spaces/rolpotamias/WiLoR/resolve/main/pretrained_models/model_config.yaml -O pretrained_models/model_config.yaml
!wget https://huggingface.co/spaces/rolpotamias/WiLoR/resolve/main/mano_data/mano_mean_params.npz -O mano_data/mano_mean_params.npz

  Cloning https://github.com/vchoutas/smplx to /tmp/pip-req-build-282z4cgk
  Running command git clone --filter=blob:none --quiet https://github.com/vchoutas/smplx /tmp/pip-req-build-282z4cgk
  Resolved https://github.com/vchoutas/smplx to commit 1265df7ba545e8b00f72e7c557c766e15c71632f
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.6/50.6 kB 2.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 50.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.8/740.8 kB 61.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 80.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 90.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 79.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 80.6 MB/s eta 0:00:00
  Created wheel for chumpy: filename=chum

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import sys
import cv2
import torch
import numpy as np
import joblib
import json
from pathlib import Path
from fastapi import FastAPI, File, UploadFile
import asyncio
import uvicorn
import nest_asyncio
from pyngrok import ngrok
from torchvision import transforms
import inspect

DRIVE_PATH = "/content/drive/MyDrive/Arabic_Sign_Language"

LETTERS_MODEL_PATH = f"{DRIVE_PATH}/models/MLP_letters.pkl"
NUMBERS_MODEL_PATH = f"{DRIVE_PATH}/models/MLP_numbers.pkl"

LETTERS_METADATA  = f"{DRIVE_PATH}/metadata/letters_metadata.json"
NUMBERS_METADATA  = f"{DRIVE_PATH}/metadata/numbers_metadata.json"
MANO_DATA_DIR     = f"{DRIVE_PATH}/mano_data"

In [4]:
!mkdir -p mano_data

!cp /content/drive/MyDrive/Arabic_Sign_Language/mano_data/MANO_RIGHT.pkl ./mano_data/
!cp /content/drive/MyDrive/Arabic_Sign_Language/mano_data/MANO_LEFT.pkl ./mano_data/
!cp /content/drive/MyDrive/Arabic_Sign_Language/mano_data/mano_mean_params.npz ./mano_data/

print("Files in local mano_data:", os.listdir("./mano_data"))

cp: cannot stat '/content/drive/MyDrive/Arabic_Sign_Language/mano_data/mano_mean_params.npz': No such file or directory
Files in local mano_data: ['MANO_LEFT.pkl', 'mano_mean_params.npz', 'MANO_RIGHT.pkl']


In [5]:
# Patch for Python 3.12 compatibility
if not hasattr(inspect, "getargspec"):
    inspect.getargspec = inspect.getfullargspec

In [6]:
# Patch for Chumpy with NumPy 1.25+ / Python 3.12
if not hasattr(np, 'int'):
    np.int = int
if not hasattr(np, 'float'):
    np.float = float
if not hasattr(np, 'complex'):
    np.complex = complex
if not hasattr(np, 'bool'):
    np.bool = bool
if not hasattr(np, 'object'):
    np.object = object
if not hasattr(np, 'str'):
    np.str = str
if not hasattr(np, 'unicode'):
    np.unicode = str

/tmp/ipykernel_10437/2934662014.py:10: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, 'object'):
/tmp/ipykernel_10437/2934662014.py:12: FutureWarning: In the future `np.str` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, 'str'):


In [7]:
sys.path.append("/content/WiLoR")
from wilor.models import load_wilor

device = "cuda" if torch.cuda.is_available() else "cpu"
wilor_model, _ = load_wilor(checkpoint_path="/content/pretrained_models/wilor_final.ckpt",
                            cfg_path="/content/pretrained_models/model_config.yaml")
wilor_model = wilor_model.to(device).eval()
print("WiLoR model loaded successfully!")

from ultralytics import YOLO
detector = YOLO("/content/pretrained_models/detector.pt")
print("YOLO detector loaded!")



/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Loading  /content/pretrained_models/wilor_final.ckpt


INFO:pytorch_lightning.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.8.1 to v2.6.1. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint pretrained_models/wilor_final.ckpt`


num_betas=10, shapedirs.shape=(778, 3, 10), self.SHAPE_SPACE_DIM=300
WiLoR model loaded successfully!
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
YOLO detector loaded!


In [8]:
letters_classifier = joblib.load(LETTERS_MODEL_PATH)
numbers_classifier = joblib.load(NUMBERS_MODEL_PATH)

with open(LETTERS_METADATA, 'r', encoding='utf-8') as f:
    letters_mapping = json.load(f)['label_mapping']

In [9]:
NUMBERS_FIX = {
    0: "0", 1: "1", 2: "10", 3: "2", 4: "3",
    5: "4", 6: "5", 7: "6", 8: "7", 9: "8", 10: "9"
}

In [10]:
wilor_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [11]:
def extract_features_api(crop_img):
    img_input = cv2.resize(crop_img, (256, 256))
    img_tensor = wilor_transform(img_input).unsqueeze(0).to(device).float()
    with torch.no_grad():
        output = wilor_model({"img": img_tensor})

    mano_params = output["pred_mano_params"]
    theta = np.concatenate([
        mano_params["global_orient"][0].cpu().numpy().flatten(),
        mano_params["hand_pose"][0].cpu().numpy().flatten()
    ])

    joints = output["pred_keypoints_3d"][0].cpu().numpy()
    hand_scale = np.linalg.norm(joints[0] - joints[9])
    tips = [4, 8, 12, 16, 20]

    dist_features = []
    for i in range(1, 5):
        dist_features.append(np.linalg.norm(joints[tips[0]] - joints[tips[i]]) / hand_scale)
    for i in range(1, 4):
        dist_features.append(np.linalg.norm(joints[tips[i]] - joints[tips[i+1]]) / hand_scale)

    return np.concatenate([theta, dist_features]).reshape(1, -1)

In [12]:
app = FastAPI()
nest_asyncio.apply()

@app.post("/predict")
async def predict(file: UploadFile = File(...), mode: str = "letters"):
    # Read Image
    contents = await file.read()
    nparr = np.frombuffer(contents, np.uint8)
    img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Hand detection
    results = detector(img_rgb, conf=0.5, verbose=False)
    if not results[0].boxes:
        return {"prediction": "No hand detected", "status": "error"}

    x1, y1, x2, y2 = results[0].boxes.xyxy[0].cpu().numpy().astype(int)
    crop = img_rgb[y1:y2, x1:x2]

    # Extract features
    features_2d = extract_features_api(crop)

    if mode == "numbers":
        pred_idx = numbers_classifier.predict(features_2d)[0]
        label = NUMBERS_FIX.get(int(pred_idx), "Unknown")
    else:
        pred_idx = letters_classifier.predict(features_2d)[0]
        label = letters_mapping.get(str(int(pred_idx)), "Unknown")
    return {"prediction": label, "status": "success"}

In [ ]:
NGROK_TOKEN = "3C7jZfpz78WFGAy8OYIDJNYZCrQ_4cehNrBESD42YZeMd7ana"
ngrok.set_auth_token(NGROK_TOKEN)
public_url = ngrok.connect(8000)
print(f"\n🚀 API is LIVE at: {public_url}")
print("Copy this URL to your Flutter app!")

config = uvicorn.Config(app, host="0.0.0.0", port=8000, loop="asyncio")
server = uvicorn.Server(config)

await server.serve()


🚀 API is LIVE at: NgrokTunnel: "https://dia-brinish-yulanda.ngrok-free.dev" -> "http://localhost:8000"
Copy this URL to your Flutter app!


INFO:     Started server process [10437]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     197.47.179.201:0 - "GET / HTTP/1.1" 404 Not Found
INFO:     197.47.179.201:0 - "GET /favicon.ico HTTP/1.1" 404 Not Found
INFO:     197.47.179.201:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     197.47.179.201:0 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     197.47.179.201:0 - "POST /predict?mode=numbers HTTP/1.1" 200 OK


/content/WiLoR/wilor/utils/geometry.py:61: UserWarning: Using torch.cross without specifying the dim arg is deprecated.
Please either pass the dim explicitly or simply use torch.linalg.cross.
The default value of dim will change to agree with that of linalg.cross in a future release. (Triggered internally at /pytorch/aten/src/ATen/native/Cross.cpp:63.)
  b3 = torch.cross(b1, b2)


INFO:     197.47.179.201:0 - "POST /predict?mode=numbers HTTP/1.1" 200 OK
INFO:     197.47.179.201:0 - "POST /predict?mode=numbers HTTP/1.1" 200 OK
